# 05주차 1번 과제 — ResNet50 기반 개/고양이/기타 분류기

Zero-shot 분류: ImageNet 사전학습 ResNet50으로 개/고양이/기타를 판별합니다.

- 추가 학습 없이 ImageNet 클래스 확률 합산 방식
- dog(151~268), cat(281~285) 인덱스 활용

In [ ]:
import torch, torchvision
print(f"torch={torch.__version__}  torchvision={torchvision.__version__}")
print(f"device={'cuda' if torch.cuda.is_available() else 'cpu'}")

## 모델 정의

In [ ]:
import torch
from pathlib import Path
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

_DOG_INDICES = set(range(151, 269))
_CAT_INDICES = set(range(281, 286))
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class DogCatClassifier:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.model.to(self.device).eval()

    def predict(self, image) -> dict:
        if not isinstance(image, Image.Image):
            image = Image.open(image).convert("RGB")
        tensor = TRANSFORM(image.convert("RGB")).unsqueeze(0).to(self.device)
        with torch.no_grad():
            probs = torch.softmax(self.model(tensor), dim=1).squeeze()
        dog_score = sum(probs[i].item() for i in _DOG_INDICES)
        cat_score = sum(probs[i].item() for i in _CAT_INDICES)
        combined = dog_score + cat_score
        if combined < 0.1:
            label, confidence = "etc", 1.0 - combined
        elif dog_score >= cat_score:
            label, confidence = "dog", dog_score / combined
        else:
            label, confidence = "cat", cat_score / combined
        return {"label": label, "confidence": round(confidence, 4),
                "dog": round(dog_score, 4), "cat": round(cat_score, 4)}


classifier = DogCatClassifier()
print(f"ResNet50 ready (device: {classifier.device})")

## 샘플 이미지 준비

CIFAR-10에서 dog/cat/etc 이미지를 각 5장씩 추출합니다.

In [ ]:
import os
from torchvision import datasets

os.makedirs("/content/samples/dog", exist_ok=True)
os.makedirs("/content/samples/cat", exist_ok=True)
os.makedirs("/content/samples/etc", exist_ok=True)

raw = datasets.CIFAR10(root="/content/data", train=False, download=True)
CIFAR_CLASSES = raw.classes

saved = {"dog": 0, "cat": 0, "etc": 0}
for img, lbl in raw:
    name = CIFAR_CLASSES[lbl]
    if name == "dog" and saved["dog"] < 5:
        img.save(f"/content/samples/dog/{saved['dog']}.png"); saved["dog"] += 1
    elif name == "cat" and saved["cat"] < 5:
        img.save(f"/content/samples/cat/{saved['cat']}.png"); saved["cat"] += 1
    elif name not in ("dog", "cat") and saved["etc"] < 5:
        img.save(f"/content/samples/etc/{saved['etc']}.png"); saved["etc"] += 1
    if all(v == 5 for v in saved.values()):
        break
print("샘플 이미지 저장 완료:", saved)

## 추론 실행

In [ ]:
def run_folder(folder: str, true_label: str):
    folder = Path(folder)
    images = [p for p in sorted(folder.iterdir()) if p.suffix.lower() in IMAGE_EXTENSIONS]
    correct = 0
    for p in images:
        r = classifier.predict(p)
        mark = "O" if r["label"] == true_label else "X"
        print(f"  [{mark}] {p.name:15s} -> {r['label'].upper():3s}  ({r['confidence']*100:.1f}%)  dog={r['dog']:.3f}  cat={r['cat']:.3f}")
        if r["label"] == true_label:
            correct += 1
    return correct, len(images)

total_c, total_n = 0, 0
for folder, label in [("/content/samples/dog", "dog"),
                       ("/content/samples/cat", "cat"),
                       ("/content/samples/etc", "etc")]:
    print(f"\n[{label.upper()}]")
    c, n = run_folder(folder, label)
    total_c += c; total_n += n
    if n: print(f"  → {c}/{n} ({c/n*100:.1f}%)")

print(f"\n전체 정확도: {total_c}/{total_n} ({total_c/total_n*100:.1f}%)")

## 커스텀 이미지 업로드 테스트

In [ ]:
from google.colab import files

print("이미지 파일을 업로드하세요 (jpg/png)")
uploaded = files.upload()

for fname, data in uploaded.items():
    with open(fname, "wb") as f:
        f.write(data)
    result = classifier.predict(fname)
    print(f"\n{fname}  →  {result['label'].upper()} ({result['confidence']*100:.1f}%)")
    print(f"  dog={result['dog']:.4f}  cat={result['cat']:.4f}")